# Intent tunability demo ; Conditioned Deep Beamformer

One network, one acquisition, a continuous control knob.

This notebook reconstructs the **PICMUS experimental resolution phantom** from a
**SPW** while sweeping the intent vector

$$q = [\,1-t,\ t\,], \qquad t \in [0, 1]$$

from `t = 0` (resolution intent) to `t = 1` (contrast intent), and measures the
lateral $-6$ dB full-width-at-half-maximum of the point targets at each step.

If intent conditioning works, lateral FWHM should increase **monotonically** with
`t`: the network is trading lateral resolution away as it is asked for contrast.

Inference only, no training, no full dataset download. Runs on CPU in a couple
of minutes, faster on a GPU.

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from scipy.signal import hilbert
import matplotlib.pyplot as plt
from pathlib import Path

DATA  = Path("data/picmus_expe_resolution_demo.npz")
CKPT  = Path("../checkpoints/conditioned_beamformer_lambda03.pt")
MODEL = Path("../beamformer/model.py")

DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PATCH_DEPTH = 128     # axial samples per network input patch
STRIDE      = 64      # 50% overlap between patches
DB_RANGE    = 60.0

print(f"device: {DEVICE}")

## 1. Load the acquisition

`picmus_expe_resolution_demo.npz` (1.6 MB) holds the **0 degree transmit only** -
one plane wave out of the 75 in the full PICMUS file - plus the reconstruction
grid and the seven annotated point-target positions.

In [ ]:
d = np.load(DATA)
rf0      = d["rf0"].astype(np.float32)      # (128 elements, n_samples)
fs, c    = float(d["fs"]), float(d["c"])
probe_x  = d["probe_x"].astype(np.float64)  # element x positions (m)
xi, zi   = d["xi"], d["zi"]                 # reconstruction grid (m)
pin_x, pin_z = d["pin_x"], d["pin_z"]       # annotated pin positions (m)

print(f"RF           : {rf0.shape[0]} elements x {rf0.shape[1]} samples "
      f"@ {fs/1e6:.2f} MHz, transmit {float(d['angle_deg']):.0f} deg")
print(f"grid         : {len(xi)} lateral x {len(zi)} axial  "
      f"({xi[0]*1e3:.1f}..{xi[-1]*1e3:.1f} mm, {zi[0]*1e3:.1f}..{zi[-1]*1e3:.1f} mm)")
print(f"point targets: {len(pin_x)}")

## 2. Load the network

In [ ]:
import importlib.util, sys

if not CKPT.exists():
    raise FileNotFoundError(
        f"Checkpoint not found at {CKPT}.\n"
        "Download conditioned_beamformer_lambda03.pt from the GitHub Release "
        "and place it in the checkpoints/ directory.")

spec = importlib.util.spec_from_file_location("bf_model", MODEL)
bf_model = importlib.util.module_from_spec(spec)
sys.modules["bf_model"] = bf_model
spec.loader.exec_module(bf_model)

model = bf_model.SmallUNetBeamformerS2DFull(n_elements=128, dropout=0.0).to(DEVICE).eval()
state = torch.load(CKPT, map_location="cpu", weights_only=True)
model.load_state_dict(state.get("model", state) if isinstance(state, dict) else state,
                      strict=True)
torch.set_grad_enabled(False)
print(f"loaded {sum(p.numel() for p in model.parameters()):,} parameters")

## 3. Reconstruction pipeline

The network predicts per-element receive apodization weights $w$; the image is
the delay-and-sum $\sum_e w_e s_e$ over the 128 elements. The steps:

1. depth gain on the input channel data
2. time-of-flight correction (TOFC) - align every element's echo to each pixel
3. network forward pass at intent $t$, giving $w$, then the weighted sum
4. envelope detection and log compression
5. resample onto the PICMUS reconstruction grid

In [ ]:
def apply_gain(rf, fs, c):
    z_cm = (0.5 * c * np.arange(rf.shape[0]) / fs) * 100.0
    g = (10.0 ** ((0.5 * 5.2 * 2.0 * z_cm) / 20.0)).astype(rf.dtype)
    return rf * g[:, None]


def compute_tofc(rf, fs, probe_x, c, device, chunk=256):
    """Time-of-flight-corrected channel data, (n_samples, n_elem, n_elem).

    Chunked over depth so it fits in modest memory on CPU; numerically identical
    to computing it in a single pass."""
    n_s, n_el = rf.shape
    rf_n = (rf - rf.mean()) / max(rf.std(), 1e-8)
    rf_t = torch.from_numpy(rf_n.astype(np.float32)).to(device)
    x = torch.from_numpy(probe_x.astype(np.float32)).to(device)
    t = torch.arange(n_s, dtype=torch.float32, device=device) / fs
    z = 0.5 * c * t
    flat = rf_t.contiguous().reshape(-1)
    col = torch.arange(n_el, device=device, dtype=torch.long)

    out = np.empty((n_s, n_el, n_el), dtype=np.float32)
    for a in range(0, n_s, chunk):
        b = min(a + chunk, n_s)
        zg, xg = torch.meshgrid(z[a:b], x, indexing="ij")
        z_f, x_f = zg.reshape(-1), xg.reshape(-1)
        dx = x_f[:, None] - x[None, :]
        rx = torch.sqrt(dx * dx + z_f[:, None] ** 2)
        td = (z_f[:, None] + rx) / c                 # transmit + receive time
        ir = torch.searchsorted(t.contiguous(), td.contiguous()).clamp(1, n_s - 1)
        il = ir - 1
        w = (td - t[il]) / (t[ir] - t[il]).clamp(min=1e-12)
        cc = col[None, :].expand(td.shape[0], -1)
        y = (1 - w) * flat[il * n_el + cc] + w * flat[ir * n_el + cc]
        y[(td < t[0]) | (td > t[-1])] = 0.0
        out[a:b] = y.reshape(b - a, n_el, n_el).cpu().numpy()
    return out


def beamform(model, tofc, t_intent, device):
    """Overlapping-patch inference; returns the stitched RF-domain image."""
    n_s, n_lat = tofc.shape[0], tofc.shape[2]
    q = torch.tensor([[1.0 - t_intent, t_intent]], dtype=torch.float32, device=device)
    accum = np.zeros((n_s, n_lat), dtype=np.float32)
    count = np.zeros(n_s, dtype=np.float32)
    for start in range(0, n_s, STRIDE):
        end = start + PATCH_DEPTH
        rows = min(end, n_s) - start
        if end > n_s:
            patch = np.pad(tofc[start:n_s], ((0, end - n_s), (0, 0), (0, 0)),
                           mode="reflect" if rows > 1 else "edge")
        else:
            patch = tofc[start:end]
        x_t = torch.from_numpy(patch.transpose(2, 0, 1)).unsqueeze(0).to(device)
        w = model(x_t, q=q)
        bf = torch.sum(w * x_t, dim=1).squeeze(0).cpu().numpy()
        accum[start:start + rows] += bf[:rows]
        count[start:start + rows] += 1.0
    return accum / np.maximum(count[:, np.newaxis], 1.0)


def to_bmode(bf, z_native, probe_x, zi, xi, pad=32):
    """Envelope + log compression, resampled onto the PICMUS grid. (n_z, n_x) dB."""
    padded = np.pad(bf, ((pad, pad), (0, 0)), mode="reflect")
    env = np.abs(hilbert(padded, axis=0))[pad:-pad].astype(np.float32)

    img = torch.from_numpy(env).unsqueeze(0).unsqueeze(0)
    z0, z1 = float(z_native[0]), float(z_native[-1])
    x0, x1 = float(probe_x[0]), float(probe_x[-1])
    zn = torch.from_numpy((2 * (zi.astype(np.float32) - z0) / max(z1 - z0, 1e-9) - 1))
    xn = torch.from_numpy((2 * (xi.astype(np.float32) - x0) / max(x1 - x0, 1e-9) - 1))
    ZZ, XX = torch.meshgrid(zn, xn, indexing="ij")
    grid = torch.stack([XX, ZZ], -1).unsqueeze(0)
    res = F.grid_sample(img, grid, mode="bilinear", align_corners=True,
                        padding_mode="border").squeeze().numpy()
    return np.clip(20 * np.log10(res / (res.max() + 1e-8) + 1e-8), -DB_RANGE, 0.0)


def lateral_fwhm_mm(db, z_idx, x_idx, x_mm, win=10):
    """-6 dB lateral width of one point target, sub-pixel interpolated."""
    lo, hi = max(0, x_idx - win), min(db.shape[1], x_idx + win + 1)
    prof, ax = db[z_idx, lo:hi], x_mm[lo:hi]
    pk = int(np.argmax(prof))
    half = prof[pk] - 6.0

    def cross(step):
        i = pk
        while 0 <= i + step < prof.size and prof[i] > half:
            i += step
        if not (0 <= i < prof.size) or prof[i] > half:
            return None
        j = i - step
        f = (prof[j] - half) / (prof[j] - prof[i] + 1e-12)
        return ax[j] + f * (ax[i] - ax[j])

    l, r = cross(-1), cross(+1)
    return abs(r - l) if (l is not None and r is not None) else np.nan


def measure_pins(db, xi, zi, pin_x, pin_z):
    x_mm = xi * 1e3
    vals = [lateral_fwhm_mm(db, int(np.argmin(np.abs(zi - pz))),
                            int(np.argmin(np.abs(xi - px))), x_mm)
            for px, pz in zip(pin_x, pin_z)]
    v = [x for x in vals if np.isfinite(x)]
    return float(np.mean(v)), float(np.std(v)), len(v)

## 4. Sweep the intent knob

The TOFC depends only on the acquisition geometry, not on intent, so it is
computed once and reused at every $t$.

In [ ]:
rf_in = apply_gain(rf0.T.astype(np.float32), fs, c)   # (n_samples, n_elements)
tofc = compute_tofc(rf_in, fs, probe_x, c, DEVICE)
z_native = 0.5 * c * np.arange(rf0.shape[1]) / fs
print(f"TOFC {tuple(tofc.shape)}")

TS = np.round(np.linspace(0.0, 1.0, 9), 3)
images, fwhm_mean, fwhm_std = {}, [], []

for t in TS:
    db = to_bmode(beamform(model, tofc, float(t), DEVICE), z_native, probe_x, zi, xi)
    images[float(t)] = db
    m, s, n = measure_pins(db, xi, zi, pin_x, pin_z)
    fwhm_mean.append(m); fwhm_std.append(s)
    print(f"  t = {t:.3f}   lateral FWHM = {m:.3f} +- {s:.3f} mm   ({n}/{len(pin_x)} pins)")

fwhm_mean, fwhm_std = np.array(fwhm_mean), np.array(fwhm_std)

## 5. Reconstructions across the intent axis

In [ ]:
show = [0.0, 0.25, 0.5, 0.75, 1.0]
ext = [xi[0] * 1e3, xi[-1] * 1e3, zi[-1] * 1e3, zi[0] * 1e3]
fig, ax = plt.subplots(1, len(show), figsize=(3.0 * len(show), 6.2))
for a, t in zip(ax, show):
    a.imshow(images[t], cmap="gray", extent=ext, aspect="auto", vmin=-DB_RANGE, vmax=0)
    a.set_title(f"$t$ = {t:g}", fontsize=11)
    a.set_xlabel("Lateral (mm)")
    if t != show[0]:
        a.set_yticks([])
ax[0].set_ylabel("Depth (mm)")
fig.suptitle("Same single plane wave across the intent axis "
             "($t=0$ resolution to $t=1$ contrast)", fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

## 6. Does the knob actually turn?

In [ ]:
fig, a = plt.subplots(figsize=(6.2, 4.2))
a.errorbar(TS, fwhm_mean, yerr=fwhm_std, fmt="o-", color="#c44e52",
           lw=2, ms=6, capsize=3)
a.set_xlabel("intent  $t$")
a.set_ylabel("lateral FWHM (mm)   lower = sharper")
a.set_title("Lateral resolution traded away as contrast is requested")
a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

rise = fwhm_mean[-1] - fwhm_mean[0]
steps = np.diff(fwhm_mean)
print(f"t=0 -> t=1 : {fwhm_mean[0]:.3f} -> {fwhm_mean[-1]:.3f} mm  ({rise:+.3f} mm)")
print(f"monotonic increasing across all {len(steps)} steps: {bool(np.all(steps >= 0))}")

## What to look for

- **Lateral FWHM rises with $t$.** The resolution intent ($t=0$) gives the
  sharpest point targets; asking for contrast broadens them. The same channel
  data and the same weights produce both - only $q$ changed.
- **The transition is gradual.** Intermediate $t$ gives intermediate resolution,
  so intent is a continuous control rather than a two-state switch.
- **The error bars overlap.** The effect is consistent in ordering but small
  relative to the spread across the seven point targets; the trend, not any
  single pair of points, is the evidence.

To see the other side of the trade-off, run the same sweep on the PICMUS
contrast phantom and measure gCNR - it moves the opposite way.